# STEP 1 - REPRODUCIBLE QUICK-CHECK

In [1]:
# Cell 1: imports and basic config
import os, json, re, ast, glob
import numpy as np, pandas as pd
from pathlib import Path

In [2]:
# Cell 2: load and inspect
CSV = "/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results/system_gnn_ae_explanations_20251005_234838.csv"

# 1.1: show what pandas thinks the header is
df = pd.read_csv(CSV, dtype=str, low_memory=False)
print("columns:", df.columns.tolist())
print("first row keys / truncated values:")
for c in df.columns:
    print(c, "->", (df[c].iloc[0] or "")[:200])

columns: ['source_node', 'target_node', 'anomaly_score']
first row keys / truncated values:
source_node -> 147
target_node -> 17913
anomaly_score -> 620.66015625


This file is an edge table with columns source_node, target_node and anomaly_score.

In [3]:
import csv
import logging

logging.basicConfig(level=logging.INFO)
ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
PATTERN = "*.csv"

EXPECTED = {"source_node","target_node","anomaly_score"}

def has_expected_header(path):
    # read only first line safely to detect header
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        first = f.readline()
    # crude split by comma, semicolon, tab guess
    for d in [',',';','\t','|']:
        cols = [c.strip() for c in first.split(d)]
        if set(EXPECTED).intersection(cols):
            return True
    return False

def read_file_flexible(path):
    # use python engine to tolerate multiline quoted fields more robustly
    try:
        df = pd.read_csv(path, engine='python', quoting=csv.QUOTE_MINIMAL, dtype=str, low_memory=False)
    except Exception as e:
        return None, f"read_error:{e}"
    # find canonical columns (exact names)
    cols = set(df.columns)
    if EXPECTED.issubset(cols):
        df = df[list(EXPECTED)]
    else:
        # try case-insensitive / trimmed matching
        col_map = {c.lower().strip(): c for c in df.columns}
        lookup = {}
        for want in EXPECTED:
            key = want.lower()
            if key in col_map:
                lookup[want] = col_map[key]
        if EXPECTED.issubset(set(lookup.keys())):
            df = df[[lookup[w] for w in ("source_node","target_node","anomaly_score")]]
            df.columns = ["source_node","target_node","anomaly_score"]
        else:
            return None, f"missing_expected_columns; available={list(df.columns)[:10]}"
    # coerce and cleanup
    df['anomaly_score'] = pd.to_numeric(df['anomaly_score'], errors='coerce')
    df['source_node'] = pd.to_numeric(df['source_node'], errors='coerce').astype('Int64')
    df['target_node'] = pd.to_numeric(df['target_node'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['source_node','target_node','anomaly_score'])
    if df.empty:
        return None, "no_numeric_rows_after_coerce"
    return df[['source_node','target_node','anomaly_score']], "ok"

# iterate files, skipping obvious non-data files
results = []
dfs = []
for p in sorted(ROOT.glob(PATTERN)):
    name = p.name
    if name.lower().startswith("files_summary") or "summary" in name.lower():
        results.append({"file": str(p), "status": "skipped_summary", "rows": 0})
        continue
    if not has_expected_header(p):
        results.append({"file": str(p), "status": "no_expected_header", "rows": 0})
        continue
    df, status = read_file_flexible(p)
    if df is None:
        results.append({"file": str(p), "status": status, "rows": 0})
    else:
        dfs.append(df)
        results.append({"file": str(p), "status": status, "rows": len(df)})
        logging.info(f"Loaded {p.name}: rows={len(df)}")

summary_df = pd.DataFrame(results)
summary_df.to_csv(ROOT / "load_attempts_verbose.csv", index=False)

if dfs:
    combined = pd.concat(dfs, ignore_index=True).drop_duplicates()
    logging.info(f"Combined shape: {combined.shape}")
else:
    logging.info("No data files loaded.")

In [4]:
# A: If you saved combined_clean.parquet earlier

PARQUET_PATH = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results/combined_clean.parquet")
if PARQUET_PATH.exists():
    combined = pd.read_parquet(PARQUET_PATH)
else:
    raise FileNotFoundError(f"{PARQUET_PATH} not found. If you already have `combined` in memory, run variant B.")

In [5]:
# B: If `combined` is already in your notebook environment, skip A and run this cell.

# ensure canonical columns
expected = {'source_node','target_node','anomaly_score'}
if not expected.issubset(set(combined.columns)):
    raise ValueError("combined must contain columns: source_node, target_node, anomaly_score")

# basic stats
print("rows:", len(combined))
print(combined['anomaly_score'].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))

# Top 50 edges by raw anomaly_score
top_edges = combined.nlargest(50, 'anomaly_score')[['source_node','target_node','anomaly_score']].reset_index(drop=True)
print("\nTop 50 edges:")
print(top_edges.to_string(index=False))
top_edges.to_csv("top_50_edges.csv", index=False)

# Node-level aggregates (source and target separately, then combined)
agg_src = combined.groupby('source_node', observed=True)['anomaly_score'].agg(['count','sum','mean','max']).rename(columns=lambda s: 'src_'+s)
agg_tgt = combined.groupby('target_node', observed=True)['anomaly_score'].agg(['count','sum','mean','max']).rename(columns=lambda s: 'tgt_'+s)

nodes = pd.concat([agg_src, agg_tgt], axis=1).fillna(0)
nodes = nodes.reset_index().rename(columns={'index':'node'})

# combined node score: max incident anomaly and total anomaly mass
nodes['combined_max'] = nodes[['src_max','tgt_max']].max(axis=1)
nodes['combined_sum'] = nodes[['src_sum','tgt_sum']].sum(axis=1)
nodes['incident_count'] = nodes[['src_count','tgt_count']].sum(axis=1)

# Top 20 nodes by combined_max (tie-breaker: combined_sum)
top_nodes = nodes.sort_values(['combined_max','combined_sum'], ascending=[False,False]).head(20)
print("\nTop 20 nodes by incident max anomaly_score:")
print(top_nodes[['node','combined_max','combined_sum','incident_count','src_count','tgt_count']].to_string(index=False))
top_nodes.to_csv("top_20_nodes.csv", index=False)

rows: 1100
count    1100.000000
mean      156.254947
std       196.473864
min        40.028343
1%         40.028343
5%         40.028343
25%        61.354324
50%        97.059013
75%       124.879318
95%       620.660034
99%       964.527527
max      1026.341064
Name: anomaly_score, dtype: float64

Top 50 edges:
 source_node  target_node  anomaly_score
         190        11632    1026.341064
       11632          190    1026.341064
         190        11373    1017.062134
       11373          190    1017.062134
         190        11633    1016.889038
       11633          190    1016.889038
       11375          190    1005.424438
         190        11375    1005.424438
       11376          190    1001.277100
         190        11376    1001.277100
         190        11636     964.527527
       11636          190     964.527527
       11374          190     963.645142
         190        11374     963.645142
       11378          190     959.026550
         190        11378     

A single node (190) dominates with 20 incident edges. combined_max = 1026.34 and combined_sum = 19728.15. Many top edges are symmetric pairs with identical scores, which suggest duplicate scoring for undirected interactions or model symmetry. There's a big plateau around 620.66 and clear clusters of very large scores around 1026 and 1017. Distribution is heavily-tailed with the 95th percentile at ~620.66 and max around 1026.34.

1. Confirm symmetry and duplicates while collapsing symmetric edges if the graph is undirected
2. Remove mirrored duplicates, keeping the higher score or 1st occurrence
3. Inspect node 190's ego edges and explanations
4. Compute degree-normalized metrics, so hubs don't dominate alerts
> incident_mean = combined_sum / incident_count
> incident_mad / z flags unsually high per-incident scores

# STEP 2 - DIAGNOSTICS

In [6]:
# assume `combined` exists with canonical columns

# 1. remove exact duplicate rows
combined = combined.drop_duplicates(subset=['source_node','target_node','anomaly_score'])

# 2. canonical undirected key to collapse mirrored edges
combined['u'] = combined[['source_node','target_node']].min(axis=1)
combined['v'] = combined[['source_node','target_node']].max(axis=1)

# keep single record per unordered edge, keep max score and count mirrors if needed
agg = combined.groupby(['u','v'], observed=True)['anomaly_score'].agg(['max','count']).reset_index().rename(columns={'max':'anomaly_score','count':'mirror_count'})

# reconstruct representative table (u->v)
edges_undirected = agg.sort_values('anomaly_score', ascending=False)

# 3. node-level aggregates after collapse
src = edges_undirected.groupby('u', observed=True)['anomaly_score'].agg(['count','sum','max']).rename(columns=lambda s: 'src_'+s)
tgt = edges_undirected.groupby('v', observed=True)['anomaly_score'].agg(['count','sum','max']).rename(columns=lambda s: 'tgt_'+s)

nodes = pd.concat([src, tgt], axis=1).fillna(0)
nodes = nodes.reset_index().rename(columns={'index':'node'})

nodes['incident_count'] = nodes[['src_count','tgt_count']].sum(axis=1)
nodes['combined_sum'] = nodes[['src_sum','tgt_sum']].sum(axis=1)
nodes['combined_max'] = nodes[['src_max','tgt_max']].max(axis=1)
nodes['incident_mean'] = nodes['combined_sum'] / nodes['incident_count'].replace(0, np.nan)
nodes = nodes.sort_values(['combined_max','incident_mean'], ascending=[False,False])

# 4. flag potential anomalies by per-incident z-score (robust)
med = nodes['incident_mean'].median()
mad = (np.abs(nodes['incident_mean'] - med)).median()
nodes['incident_mean_z'] = (nodes['incident_mean'] - med) / (mad if mad>0 else nodes['incident_mean'].std())
nodes['flag_incident_mean'] = nodes['incident_mean_z'] > 6

# quick outputs
print("Top nodes by combined_max:")
print(nodes[['node','combined_max','combined_sum','incident_count','incident_mean','incident_mean_z']].head(20).to_string(index=False))
print("\nTop undirected edges:")
print(edges_undirected.head(50).to_string(index=False))

# 5. ego edge inspection for node 190 (list edges and source files to trace explanations)
ego = edges_undirected[(edges_undirected['u']==190) | (edges_undirected['v']==190)].sort_values('anomaly_score', ascending=False)
print("\nEgo edges for node 190:")
print(ego.to_string(index=False))

Top nodes by combined_max:
 node  combined_max  combined_sum  incident_count  incident_mean  incident_mean_z
11632   1026.341064   1026.341064             1.0    1026.341064        29.913956
  190   1026.341064   9864.073364            10.0     986.407336        28.628474
11373   1017.062134   1017.062134             1.0    1017.062134        29.615264
11633   1016.889038   1016.889038             1.0    1016.889038        29.609692
11375   1005.424438   1005.424438             1.0    1005.424438        29.240642
11376   1001.277100   1001.277100             1.0    1001.277100        29.107137
11636    964.527527    964.527527             1.0     964.527527        27.924154
11374    963.645142    963.645142             1.0     963.645142        27.895750
11378    959.026550    959.026550             1.0     959.026550        27.747075
11377    955.081177    955.081177             1.0     955.081177        27.620072
11380    954.799194    954.799194             1.0     954.799194       

In [7]:
# A: collapse mirrored edges and compute node aggregates (robust)

# assume `combined` DataFrame exists; otherwise load the parquet
ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
# combined = pd.read_parquet(ROOT / "combined_clean.parquet")  # optional load

# drop exact duplicates and create undirected keys
combined = combined.drop_duplicates(subset=['source_node','target_node','anomaly_score']).copy()
combined['u'] = combined[['source_node','target_node']].min(axis=1)
combined['v'] = combined[['source_node','target_node']].max(axis=1)

# keep max score per undirected pair and count mirrors
edges_undirected = combined.groupby(['u','v'], observed=True)['anomaly_score'].agg(['max','count']).reset_index().rename(columns={'max':'anomaly_score','count':'mirror_count'})
edges_undirected = edges_undirected.sort_values('anomaly_score', ascending=False).reset_index(drop=True)

# recompute node aggregates on collapsed edges
src = edges_undirected.groupby('u', observed=True)['anomaly_score'].agg(['count','sum','max']).rename(columns=lambda s: 'src_'+s)
tgt = edges_undirected.groupby('v', observed=True)['anomaly_score'].agg(['count','sum','max']).rename(columns=lambda s: 'tgt_'+s)
nodes = pd.concat([src, tgt], axis=1).fillna(0)
nodes = nodes.reset_index().rename(columns={'index':'node'})
nodes['incident_count'] = nodes[['src_count','tgt_count']].sum(axis=1)
nodes['combined_sum'] = nodes[['src_sum','tgt_sum']].sum(axis=1)
nodes['combined_max'] = nodes[['src_max','tgt_max']].max(axis=1)
nodes['incident_mean'] = nodes['combined_sum'] / nodes['incident_count'].replace(0, np.nan)

# robust z using MAD
med = nodes['incident_mean'].median()
mad = (np.abs(nodes['incident_mean'] - med)).median()
nodes['incident_mean_z'] = (nodes['incident_mean'] - med) / (mad if mad>0 else nodes['incident_mean'].std())
nodes = nodes.sort_values(['combined_max','incident_mean'], ascending=[False,False]).reset_index(drop=True)

# outputs
edges_undirected.head(50).to_csv(ROOT / "top_50_undirected_edges.csv", index=False)
nodes.head(50).to_csv(ROOT / "top_50_nodes_collapsed.csv", index=False)
print("Saved top_50_undirected_edges.csv and top_50_nodes_collapsed.csv")
edges_undirected.head(10)

Saved top_50_undirected_edges.csv and top_50_nodes_collapsed.csv


,u,v,anomaly_score,mirror_count
0,190,11632,1026.341064,2
1,190,11373,1017.062134,2
2,190,11633,1016.889038,2
3,190,11375,1005.424438,2
4,190,11376,1001.277100,2
5,190,11636,964.527527,2
6,190,11374,963.645142,2
7,190,11378,959.026550,2
8,190,11377,955.081177,2
9,190,11380,954.799194,2


In [8]:
# B: map undirected edges back to filenames that contain them
import csv

# build a lookup of string patterns we want to find (u,v and v,u)
top_n = 200  # adjust
targets = set()
for _, row in edges_undirected.head(top_n).iterrows():
    u, v = int(row['u']), int(row['v'])
    targets.add(f"{u},{v}")
    targets.add(f"{v},{u}")

# scan CSV files and record which files contain which pattern (line-level search)
ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
files = sorted(ROOT.glob("*.csv"))
edge_file_map = {}  # pattern -> set(files)
for p in files:
    try:
        text = p.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        continue
    for pat in list(targets):
        if pat in text:
            edge_file_map.setdefault(pat, []).append(str(p.name))

# convert to DataFrame for review
rows = []
for _, row in edges_undirected.head(top_n).iterrows():
    u, v = int(row['u']), int(row['v'])
    pat1, pat2 = f"{u},{v}", f"{v},{u}"
    files_for = sorted(set(edge_file_map.get(pat1, []) + edge_file_map.get(pat2, [])))
    rows.append({"u":u,"v":v,"anomaly_score":row['anomaly_score'],"files": ";".join(files_for)})
map_df = pd.DataFrame(rows)
map_df.to_csv(ROOT / "edge_to_files_map_topN.csv", index=False)
print(f"Saved edge_to_files_map_topN.csv with {len(map_df)} rows")
map_df.head(20)

Saved edge_to_files_map_topN.csv with 200 rows


,u,v,anomaly_score,files
0,190,11632,1026.341064,edge_explanations_prioritized_summary.csv;edge...
1,190,11373,1017.062134,edge_explanations_prioritized_summary.csv;edge...
2,190,11633,1016.889038,edge_explanations_prioritized_summary.csv;edge...
3,190,11375,1005.424438,edge_explanations_prioritized_summary.csv;edge...
4,190,11376,1001.277100,edge_explanations_prioritized_summary.csv;edge...
5,190,11636,964.527527,edge_explanations_prioritized_summary.csv;edge...
6,190,11374,963.645142,edge_explanations_prioritized_summary.csv;edge...
7,190,11378,959.026550,edge_explanations_prioritized_summary.csv;edge...
8,190,11377,955.081177,edge_explanations_prioritized_summary.csv;edge...
9,190,11380,954.799194,edge_explanations_prioritized_summary.csv;edge...


In [9]:
# C: extract explanation text for a selected list of (u,v) pairs into CSV
import re

ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
# choose edges to extract (example: top 50 from map_df)
extract_list = map_df.head(50)[['u','v']].to_dict(orient='records')

out_rows = []
for pair in extract_list:
    u, v = pair['u'], pair['v']
    patterns = [f"{u},{v}", f"{v},{u}"]
    for p in files:
        try:
            # fast skip using previously built map if available
            if not any(p.name in (edge_file_map.get(pt, []) or []) for pt in patterns):
                continue
            # stream file to avoid memory blowup and find matching lines
            with open(p, 'r', encoding='utf-8', errors='ignore') as fh:
                for line in fh:
                    # crude CSV line match; handles quoted fields since we search raw text
                    if any(pt in line for pt in patterns):
                        # attempt to parse CSV row robustly to capture explanation
                        try:
                            # use csv module to handle quoted commas
                            row = next(csv.reader([line]))
                        except StopIteration:
                            continue
                        # try to assign columns if header known; else fallback by position
                        # We expect source_node,target_node,anomaly_score,explanation or at least first three cols
                        if len(row) >= 4:
                            src, tgt, score, explanation = row[0], row[1], row[2], ",".join(row[3:])
                        elif len(row) == 3:
                            src, tgt, score = row
                            explanation = ""
                        else:
                            continue
                        if str(src).strip() in {str(u), str(v)} and str(tgt).strip() in {str(u), str(v)}:
                            out_rows.append({
                                "u": u, "v": v, "file": p.name,
                                "source_node": src.strip(), "target_node": tgt.strip(),
                                "anomaly_score": float(score) if score.replace('.','',1).isdigit() else None,
                                "explanation": explanation.strip()
                            })
        except Exception:
            continue

out_df = pd.DataFrame(out_rows)
out_df.to_csv(ROOT / "edge_explanations_top50.csv", index=False)
print("Saved edge_explanations_top50.csv with", len(out_df), "rows")
out_df.head(20)

Saved edge_explanations_top50.csv with 550 rows


,u,v,file,source_node,target_node,anomaly_score,explanation
0,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,,190,11632,1026.341064453125,{'node_..."
1,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,,11632,190,1026.341064453125,{'node_..."
2,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,,190,11632,1026.341064453125,2"
3,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,1026.341064453125,edge_explanations_..."
4,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,1026.341064453125,{'node_feat_mask':..."
5,190,11632,edge_explanations_top50.csv,190,11632,NaN,"11632,190,1026.341064453125,{'node_feat_mask':..."
6,190,11632,edge_explanations_top50.csv,190,11632,NaN,"190,11632,1026.341064453125,2"
7,190,11632,edge_to_files_map_topN.csv,190,11632,1026.341064,edge_explanations_prioritized_summary.csv;edge...
8,190,11632,system_gnn_ae_explanations_20251006_003952.csv,190,11632,1026.341064,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0...."
9,190,11632,system_gnn_ae_explanations_20251006_003952.csv,11632,190,1026.341064,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0...."


The extractor found multiple identical high scores that concentrate on node 190 along with symmetric pairs. The CSVs contain a multiline explanation that repeats for these top pairs. edge_explanations_top50.csv has 150 rows, because each undirected pair appears twice as (u,v and then v,u) along with the mapping rows. Many explanation texts look identical, so a repeated model artifact or duplicated logging is present.

In [10]:
# 1) check how many unique explanation texts per undirected edge
fn = ROOT / "edge_explanations_top50.csv"
df = pd.read_csv(fn, dtype=str)
df['pair'] = df[['u','v']].astype(str).agg(','.join, axis=1)
uniq_per_pair = df.groupby('pair')['explanation'].nunique().sort_values(ascending=False)
print(uniq_per_pair.head(20))

# 2) quick hashing to detect exact duplicates
import hashlib
df['ex_hash'] = df['explanation'].fillna('').apply(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())
hash_counts = df.groupby(['pair','ex_hash']).size().reset_index(name='count').sort_values(['pair','count'], ascending=[True,False])
print(hash_counts.head(40))

# 3) collapse mirrored edges (representative row per undirected pair)
edges = pd.read_csv(fn)
edges = edges.drop_duplicates(subset=['u','v']).sort_values('anomaly_score', ascending=False).reset_index(drop=True)
edges.head(50).to_csv("top_50_collapsed.csv", index=False)

pair
255,6865     11
253,6875     11
240,20237    11
283,7692     11
298,11788    11
272,1200     11
265,8876     11
266,7690     11
314,20284    11
322,7734     11
281,20283    11
265,7632     11
267,8872     11
190,11373    10
190,11378    10
190,11375    10
190,11633    10
190,11374    10
278,5277     10
254,11720    10
Name: explanation, dtype: int64
         pair                                            ex_hash  count
8   147,17913  e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...      2
0   147,17913  04b789c25bddd860f8f8d4adad7a99f37e2e3a33160f22...      1
1   147,17913  115e81904802819f9a1d5cf6d1da71dd4feda0b15a4166...      1
2   147,17913  3b405b435590424a0b51916492b8246f0c68795a2c0e21...      1
3   147,17913  4f3db0b9501d0c82c6fdb1c36ceb5c45a2df0b6c4cb5d3...      1
4   147,17913  65519ce5d2248cea7a5f0d4964f7eef250fd91bf5a6e76...      1
5   147,17913  73b7bd35103dc6375f173ec9d307d7dcb406d0ab5da6db...      1
6   147,17913  b4c0604df700a0830b0f744b2057547b8d322382c2b1ea...   

Many undirected pairs repeatedly appear across files, notably pairs involving 190. The computed SHA-256 hashes of explanation texts show that some pairs have a single repeated hash, while others have multiple different hashes. The hash e3b0c442... is for an empty string, so pair 147,17913 has at least one empty or missing explanation and one or more non-empty explanations. The hash 858397bf... shows up repeatedly across many top pairs, suggesting a recurring explanation pattern. 190 additionally has multiple different hashes, so some repeated explanation content or distinct explanation variants exist for the same edge.

In [11]:
ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
df = pd.read_csv(ROOT / "edge_explanations_top50.csv", dtype=str)
df['pair'] = df[['u','v']].astype(str).agg(','.join, axis=1)
df['explanation'] = df['explanation'].fillna('').astype(str)
import hashlib
df['ex_hash'] = df['explanation'].apply(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())

# summary
hash_counts = df.groupby('pair')['ex_hash'].nunique().reset_index(name='unique_explanation_count')
top_hash = df.groupby(['pair','ex_hash']).size().reset_index(name='count').sort_values(['pair','count'], ascending=[True,False]).groupby('pair').first().reset_index()
files = df.groupby('pair')['file'].agg(lambda s: ";".join(sorted(set(s)))).reset_index(name='files')
max_score = df.groupby('pair')['anomaly_score'].agg(lambda s: float(s.astype(float).max())).reset_index(name='max_anomaly_score')

summary = hash_counts.merge(top_hash, on='pair').merge(files, on='pair').merge(max_score, on='pair')
summary = summary.rename(columns={'ex_hash':'top_ex_hash','count':'top_count'})
summary['empty_present'] = summary['top_ex_hash'].apply(lambda h: h == hashlib.sha256(b'').hexdigest()) | df.groupby('pair')['ex_hash'].apply(lambda hs: hashlib.sha256(b'').hexdigest() in set(hs)).values
summary.to_csv(ROOT / "edge_explanations_prioritized_summary.csv", index=False)
print("Saved edge_explanations_prioritized_summary.csv")
print(summary.sort_values(['max_anomaly_score','unique_explanation_count'], ascending=[False,False]).head(20))

Saved edge_explanations_prioritized_summary.csv
         pair  unique_explanation_count  \
8   190,11632                        10   
1   190,11373                        10   
9   190,11633                        10   
3   190,11375                        10   
4   190,11376                        10   
10  190,11636                        10   
2   190,11374                        10   
6   190,11378                        10   
5   190,11377                        10   
7   190,11380                        10   
11  218,12241                        10   
20  259,11453                        10   
33   278,5277                        10   
14  250,11451                        10   
22   260,8554                        10   
12  236,11450                        10   
43   300,5293                        10   
30   272,1200                        11   
45   304,7405                        10   
42  299,18294                        10   

                                          top_ex

In [12]:
# Representative explanation per pair -> CSV
import hashlib

ROOT = Path("/content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results")
IN = ROOT / "edge_explanations_top50.csv"
OUT = ROOT / "edge_explanations_representative_top50.csv"

# load
df = pd.read_csv(IN, dtype=str)
df[['u','v']] = df[['u','v']].astype(int)
df['pair'] = df[['u','v']].astype(str).agg(','.join, axis=1)
df['explanation'] = df['explanation'].fillna('').astype(str)
df['ex_hash'] = df['explanation'].apply(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())

# count hashes per pair and pick the most frequent hash (tie -> first by appearance)
hash_counts = df.groupby(['pair','ex_hash']).size().reset_index(name='count')
hash_counts = hash_counts.sort_values(['pair','count'], ascending=[True,False])
top_hash_per_pair = hash_counts.groupby('pair', as_index=False).first().rename(columns={'ex_hash':'top_ex_hash','count':'top_count'})

# join back to pick a representative row (first occurrence of top_ex_hash per pair)
rep = top_hash_per_pair.merge(df, left_on=['pair','top_ex_hash'], right_on=['pair','ex_hash'], how='left')
rep = rep[['pair','u','v','top_ex_hash','top_count','file','anomaly_score','explanation']].drop_duplicates('pair').reset_index(drop=True)

# mark empty explanations
empty_hash = hashlib.sha256(b'').hexdigest()
rep['empty_explanation'] = rep['top_ex_hash'] == empty_hash

# save compact CSV (truncates long explanation visually but preserves full text in file)
rep.to_csv(OUT, index=False)
print("Saved representative explanations to:", OUT)
rep.head(50)

Saved representative explanations to: /content/drive/MyDrive/CS/AI Cybersecurity/CIS_datasets/AnomalyDetectionResults/testbed_system_1_20251005_train_results/edge_explanations_representative_top50.csv


,pair,u,v,top_ex_hash,top_count,file,anomaly_score,explanation,empty_explanation
0,"147,17913",147,17913,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,2,system_gnn_ae_explanations_20251005_234838.csv,620.66015625,,True
1,"190,11373",190,11373,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,1017.0621337890625,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
2,"190,11374",190,11374,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,963.6451416015625,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
3,"190,11375",190,11375,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,1005.4244384765625,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
4,"190,11376",190,11376,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,1001.277099609375,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
5,"190,11377",190,11377,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,955.0811767578125,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
6,"190,11378",190,11378,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,959.0265502929688,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
7,"190,11380",190,11380,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,954.7991943359375,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
8,"190,11632",190,11632,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,1026.341064453125,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False
9,"190,11633",190,11633,858397bf798a39398f2c6ed7fe128289ca365bc439d79e...,2,system_gnn_ae_explanations_20251006_003952.csv,1016.8890380859375,"{'node_feat_mask': tensor([[0.0000, 0.0000, 0....",False


Here's a file that shows a presentative explanation per undirected pair for human review.

# Summary

Dataset = edge table files with columns of source_node, target_node, anomaly_score and an exra explanation text field in some files
<br>
<br>
Rows loaded = 1,100 edges across files
<br>
<br>
Distribution = heavily-tailed
<br>
95th percentile = ~620.66
Max = ~1026.34
<br>
<br>
Primary finding = node 190 dominates due to many high-scoring incident edges; many top edges appear as symmetric pairs; explanations show repeeated and some distinct texts

## Tasks

1. Check recurring hash across many pairs, is it default/placeholder mask?
2. For many explanations identical across files, mark those edges and see if they convey real signal
3. Validate node-level normalization and triage
> Degree-normalized score
> Review flagged nodes
> Check edge-candidates
4. Manually inspect rpresentative explanations for a stratified sample